# Deteccion de Outliers y Anomalias

_Detectar valores extremos y anomalias como parte de la limpieza de datos: metodos estadisticos (z-score, IQR, MAD) y basados en ML (Isolation Forest, DBSCAN)._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Deteccion de Outliers](assets/header.png)


## Introduccion

Un **outlier** es una observacion que se aparta notablemente del grueso de los
datos. Pueden ser **errores** (un sensor que falla, un dedo gordo al teclear) o
**senales reales pero raras** (fraude, una transaccion legitima enorme). En
feature engineering importan por dos razones:

- **Distorsionan estadisticos y modelos.** La media, la desviacion estandar, la
  regresion lineal o el escalado min-max son *muy* sensibles a valores extremos: un
  solo outlier puede arrastrar la media o estirar el rango.
- **Limpiar datos es parte del pipeline.** Antes de transformar y escalar conviene
  decidir que hacer con los extremos: dejarlos, recortarlos o transformarlos.

Veremos dos familias de metodos para *detectarlos* y luego *que hacer* con ellos:

1. **Metodos estadisticos** (univariados): z-score, regla IQR / vallas de Tukey,
   z-score modificado con MAD.
2. **Metodos basados en ML** (multivariados): Isolation Forest y DBSCAN
   (deteccion por densidad).
3. **Que hacer con los outliers**: eliminar, recortar / winsorizar, o transformar.

> **Outlier vs anomalia.** A escala univariada solemos hablar de *outliers*
> (valores extremos en una columna); cuando la rareza es *multivariada* (una
> combinacion inusual de varias variables, aunque cada una por separado sea normal)
> hablamos de *anomalias*, y ahi brillan los metodos basados en ML.


## Setup y datos

Reusamos el Titanic (con fallback sintetico). Las columnas `age` y `fare` tienen
colas y valores extremos reales, ideales para ilustrar la deteccion de outliers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)

def load_titanic():
    try:
        import seaborn as sns
        df = sns.load_dataset("titanic")
        if df is None or len(df) == 0:
            raise ValueError("vacio")
        print("Dataset real del Titanic cargado desde seaborn.")
        return df
    except Exception as e:
        print(f"Usando datos sinteticos ({type(e).__name__}: {e})")
        n = 500
        return pd.DataFrame({
            "survived": np.random.randint(0, 2, n),
            "pclass": np.random.choice([1, 2, 3], n, p=[0.2, 0.2, 0.6]),
            "age": np.clip(np.random.normal(29, 14, n), 0.4, 80).round(1),
            "fare": np.round(np.random.exponential(32, n), 4),
        })

df = load_titanic()
fare = df["fare"].dropna().to_numpy(dtype=float)
age = df["age"].dropna().to_numpy(dtype=float)
print("fare:", fare.shape, "| age:", age.shape)
df[["age", "fare"]].describe()

## 1. Metodos estadisticos (univariados)

### 1a. Regla del z-score

**Intuicion primero.** Estandarizamos cada valor (cuantas desviaciones estandar se
aleja de la media) y marcamos como outlier todo lo que cae mas alla de un umbral,
tipicamente $|z| > 3$.

$$z_i = \frac{x_i - \mu}{\sigma}, \qquad \text{outlier si } |z_i| > 3.$$

**Cuidado.** La media $\mu$ y la desviacion $\sigma$ son *ellas mismas* sensibles a
los outliers: unos pocos extremos inflan $\sigma$ y "esconden" a los demas. Sirve
bien cuando los datos son aproximadamente **normales** y los extremos son pocos.

In [ ]:
mu, sigma = fare.mean(), fare.std()
z = (fare - mu) / sigma
out_z = np.abs(z) > 3
print(f"fare: media={mu:.2f}  std={sigma:.2f}")
print(f"z-score (|z|>3): {out_z.sum()} outliers de {len(fare)}")
print("valores marcados:", np.sort(fare[out_z])[-8:].round(2))

### 1b. Regla IQR / vallas de Tukey

**Intuicion primero.** En vez de media y desviacion (sensibles), usamos
**cuartiles**, que son robustos. El **rango intercuartil** $\text{IQR} = Q_3 - Q_1$
mide la dispersion del 50% central. Marcamos como outlier todo lo que cae fuera de
las **vallas de Tukey**:

$$[\,Q_1 - 1.5\,\text{IQR}, \;\; Q_3 + 1.5\,\text{IQR}\,].$$

(Con $3\,\text{IQR}$ en lugar de $1.5$ se marcan solo los outliers "extremos".) Es
el criterio que dibuja el clasico **boxplot**: los bigotes llegan hasta las vallas
y los puntos mas alla se grafican como outliers.

In [ ]:
def iqr_fences(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

low, high = iqr_fences(fare)
out_iqr = (fare < low) | (fare > high)
print(f"Q1={np.percentile(fare,25):.2f}  Q3={np.percentile(fare,75):.2f}  "
      f"IQR={np.percentile(fare,75)-np.percentile(fare,25):.2f}")
print(f"Vallas de Tukey: [{low:.2f}, {high:.2f}]")
print(f"IQR (1.5x): {out_iqr.sum()} outliers de {len(fare)}")

In [ ]:
# Boxplot: los puntos mas alla de los bigotes son los outliers segun la regla IQR.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot(fare, vert=True, widths=0.5)
ax[0].set_title("Boxplot de 'fare'\n(puntos = outliers por IQR)"); ax[0].set_ylabel("fare")

# Scatter resaltando los outliers IQR sobre el indice.
ax[1].scatter(np.arange(len(fare))[~out_iqr], fare[~out_iqr], s=10,
              c="#4c78a8", label="normal")
ax[1].scatter(np.arange(len(fare))[out_iqr], fare[out_iqr], s=28,
              c="#e45756", label="outlier IQR")
ax[1].axhline(high, ls="--", c="gray", lw=1, label="valla superior")
ax[1].set_title("'fare' con outliers resaltados"); ax[1].set_xlabel("indice")
ax[1].set_ylabel("fare"); ax[1].legend()
plt.tight_layout(); plt.show()

### 1c. z-score modificado (con MAD)

**Intuicion primero.** Arreglamos la fragilidad del z-score clasico cambiando media
y desviacion por sus versiones **robustas**: la **mediana** y la **MAD** (mediana de
las desviaciones absolutas respecto a la mediana).

$$\text{MAD} = \text{mediana}\big(|x_i - \tilde{x}|\big), \qquad
M_i = \frac{0.6745\,(x_i - \tilde{x})}{\text{MAD}}.$$

El factor $0.6745$ hace que $M$ sea comparable a un z-score bajo normalidad. Una
regla comun: outlier si $|M_i| > 3.5$. Como la mediana y la MAD apenas se mueven
con los extremos, este criterio **no se deja enganar** por los propios outliers.

In [ ]:
def modified_zscore(x):
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad == 0:
        return np.zeros_like(x, dtype=float)
    return 0.6745 * (x - med) / mad

m = modified_zscore(fare)
out_mad = np.abs(m) > 3.5
print(f"mediana={np.median(fare):.2f}  MAD={np.median(np.abs(fare-np.median(fare))):.2f}")
print(f"z-score modificado (|M|>3.5): {out_mad.sum()} outliers")

# Comparacion de los tres criterios univariados.
print("\nComparacion de conteos sobre 'fare':")
print(f"  z-score clasico  (|z|>3)  : {out_z.sum()}")
print(f"  IQR / Tukey      (1.5xIQR): {out_iqr.sum()}")
print(f"  z-score modif.   (|M|>3.5): {out_mad.sum()}")

## 2. Metodos basados en ML (multivariados)

Los criterios anteriores miran **una columna a la vez**. Pero una observacion puede
ser normal en cada variable por separado y aun asi ser una **combinacion** rara.
Para eso usamos metodos multivariados. Construimos un set 2-D etiquetado con
**outliers inyectados** (asi conocemos la verdad y podemos medir ROC-AUC).

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.metrics import roc_auc_score

rng = np.random.RandomState(42)
n_inliers, n_outliers = 500, 50

X_in, _ = make_blobs(n_samples=n_inliers, centers=[[2, 2], [-2, -2]],
                     cluster_std=0.6, random_state=42)
X_out = rng.uniform(low=-6, high=6, size=(n_outliers, 2))

X = np.vstack([X_in, X_out])
y_true = np.hstack([np.zeros(n_inliers), np.ones(n_outliers)]).astype(int)  # 1=anomalia
contamination = n_outliers / (n_inliers + n_outliers)
print(f"X: {X.shape} | inliers: {n_inliers} | outliers: {n_outliers} | "
      f"contaminacion: {contamination:.3f}")

plt.figure(figsize=(6, 5))
plt.scatter(X[y_true == 0, 0], X[y_true == 0, 1], s=14, c="tab:blue", label="inlier")
plt.scatter(X[y_true == 1, 0], X[y_true == 1, 1], s=24, c="tab:red", marker="x", label="anomalia")
plt.legend(); plt.title("Dataset 2-D etiquetado (outliers uniformes inyectados)")
plt.tight_layout(); plt.show()

### 2a. Isolation Forest

**Intuicion primero.** Las anomalias son *pocas y distintas*, asi que son **faciles
de aislar**. Un *iTree* particiona los datos eligiendo una **variable aleatoria** y
un **corte aleatorio**. Las anomalias, en regiones esparsas, quedan separadas tras
**muy pocos cortes** (camino corto desde la raiz); los inliers, en regiones densas,
necesitan **muchos** cortes (camino largo).

El score usa la **longitud de camino esperada** $E(h(x))$ sobre el ensemble,
normalizada por la longitud media de una busqueda fallida en un BST de $n$ puntos,

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}, \qquad H(i)\approx \ln(i) + 0.5772,$$

dando

$$s(x, n) = 2^{-\,E(h(x)) / c(n)}.$$

$s \to 1$: caminos cortos → **anomalia**; $s \to 0.5$: cerca de la media → normal.
Es rapido ($O(n\log n)$), maneja alta dimension y **no computa distancias**.

In [ ]:
from sklearn.ensemble import IsolationForest

iforest = IsolationForest(n_estimators=200, contamination=contamination,
                          random_state=42)
iforest.fit(X)
# decision_function: mayor = mas normal. Lo negamos para que mayor = mas anomalo.
if_scores = -iforest.decision_function(X)
if_auc = roc_auc_score(y_true, if_scores)
print(f"Isolation Forest ROC-AUC: {if_auc:.4f}")

### 2b. DBSCAN (deteccion por densidad)

**Intuicion primero.** DBSCAN agrupa los puntos en regiones **densas** y deja
fuera lo que no encaja: los puntos que no pertenecen a ningun cluster se etiquetan
como **ruido** (`label = -1`), y ese ruido son justamente nuestros **outliers**. No
hay que decirle cuantos clusters hay — la **densidad** lo decide.

Dos hiperparametros mandan:

- **`eps`**: radio del vecindario. Un punto es **nucleo (core)** si tiene al menos
  `min_samples` vecinos dentro de `eps`.
- **`min_samples`**: cuantos vecinos definen una region densa.

Un punto que no es nucleo ni **alcanzable** desde un nucleo queda como
**ruido / outlier**. DBSCAN encuentra anomalias **globales y de forma arbitraria**
(no asume blobs gaussianos), pero es sensible a la **escala** (estandariza antes) y
sobre todo a `eps`.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# DBSCAN mide distancias -> estandarizamos primero.
X_std = StandardScaler().fit_transform(X)
dbscan = DBSCAN(eps=0.5, min_samples=8).fit(X_std)
labels = dbscan.labels_                 # -1 = ruido (outlier)
db_outlier = (labels == -1)

# DBSCAN da una etiqueta DURA (no un score): usamos el flag binario como "score".
db_auc = roc_auc_score(y_true, db_outlier.astype(float))
print(f"DBSCAN — outliers (ruido) detectados: {db_outlier.sum()}  | ROC-AUC: {db_auc:.4f}")

plt.figure(figsize=(6.5, 5.5))
plt.scatter(X[~db_outlier, 0], X[~db_outlier, 1], s=14, c=labels[~db_outlier],
            cmap="tab10", label="clusters densos")
plt.scatter(X[db_outlier, 0], X[db_outlier, 1], s=45, c="red", marker="x",
            label="ruido = outlier")
plt.legend(loc="upper left"); plt.title("DBSCAN: los puntos de ruido (-1) son outliers")
plt.tight_layout(); plt.show()

### Vista comparada: Isolation Forest vs DBSCAN

A la izquierda, los contornos del **score** de Isolation Forest con su **frontera**
(linea discontinua). A la derecha, los **clusters densos** de DBSCAN con el ruido
(`-1`) resaltado. IF entrega un *score continuo* por punto; DBSCAN, una *etiqueta
dura* (cluster vs ruido).

In [ ]:
xx, yy = np.meshgrid(np.linspace(-7, 7, 400), np.linspace(-7, 7, 400))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
# Izquierda: Isolation Forest (score + frontera)
Z = iforest.decision_function(grid).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, levels=25, cmap="RdBu")
axes[0].contour(xx, yy, Z, levels=[0], colors="k", linestyles="dashed", linewidths=1.5)
axes[0].scatter(X[y_true == 0, 0], X[y_true == 0, 1], s=10, c="white",
                edgecolor="k", lw=0.3, label="inlier")
axes[0].scatter(X[y_true == 1, 0], X[y_true == 1, 1], s=30, c="yellow",
                edgecolor="k", marker="x", label="anomalia")
axes[0].set_title("Isolation Forest — score y frontera")
axes[0].legend(loc="upper left", fontsize=8)
# Derecha: DBSCAN (clusters densos vs ruido)
axes[1].scatter(X[~db_outlier, 0], X[~db_outlier, 1], s=12,
                c=labels[~db_outlier], cmap="tab10")
axes[1].scatter(X[db_outlier, 0], X[db_outlier, 1], s=45, c="red", marker="x",
                label="ruido = outlier")
axes[1].set_title("DBSCAN — clusters densos vs ruido (-1)")
axes[1].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

### Comparativa y cuando usar cada metodo ML

| Aspecto | **Isolation Forest** | **DBSCAN** |
|---|---|---|
| Principio | aislamiento aleatorio, longitud de camino | densidad: nucleos, alcanzabilidad y ruido |
| Salida | **score continuo** de anomalia | **etiqueta dura** (cluster vs ruido `-1`) |
| Tipo de outlier | global, multivariado | global, **forma arbitraria** (no asume blobs) |
| Tamano de datos | **grande**, $O(n\log n)$ | pequeno/medio (coste de vecindarios) |
| Knobs clave | `n_estimators`, `contamination` | `eps`, `min_samples` (y **escala**) |

**Reglas de oro**

- **Datos grandes / alta dimension, baseline rapido y con score →** Isolation Forest.
- **Clusteres de densidad o forma arbitraria, sin fijar cuantos hay →** DBSCAN
  (estandariza y ajusta `eps`).

In [ ]:
comparison = pd.DataFrame({
    "IsolationForest": {"roc_auc": if_auc},
    "DBSCAN": {"roc_auc": db_auc},
}).T.sort_values("roc_auc", ascending=False)
print("ROC-AUC sobre los outliers inyectados conocidos:")
print(comparison.round(4))

## 3. Que hacer con los outliers

Detectarlos es solo la mitad: hay que **decidir que hacer**. Tres estrategias
principales, con criterios para elegir:

- **Eliminar (drop).** Quita las filas marcadas. Solo si estas convencido de que son
  **errores** y son **pocas**; si son senal real (fraude), eliminarlas borra
  justamente lo que quieres detectar. Tirar muchas filas sesga el dataset.
- **Recortar / winsorizar (cap).** En vez de borrar, **limitas** los valores
  extremos a un percentil (p. ej. todo por encima del P99 pasa a valer el P99).
  Conserva la fila y su informacion, solo domestica la magnitud. Buen default
  cuando los extremos son reales pero distorsionan.
- **Transformar.** Aplicar `log` / Box-Cox (Notebook 1) **comprime las colas**, asi
  que muchos "outliers" dejan de serlo sin borrar nada. Ideal para variables
  positivas y sesgadas como `fare`.

La eleccion depende de *por que* existe el outlier (error vs senal), *cuantos* hay
y del *modelo* (los arboles toleran extremos; los lineales/distancia, no).

In [ ]:
# WINSORIZAR: capar 'fare' a los percentiles 1 y 99.
p1, p99 = np.percentile(fare, [1, 99])
fare_wins = np.clip(fare, p1, p99)
print(f"Winsorizado a [P1={p1:.2f}, P99={p99:.2f}]")
print(f"max antes={fare.max():.2f}  ->  max despues={fare_wins.max():.2f}")

# TRANSFORMAR: log1p comprime la cola (muchos outliers dejan de serlo).
fare_log = np.log1p(fare)
low_l, high_l = iqr_fences(fare_log)
out_log = (fare_log < low_l) | (fare_log > high_l)
print(f"\nOutliers IQR antes de log : {out_iqr.sum()}")
print(f"Outliers IQR tras log1p   : {out_log.sum()}  (la transformacion los reduce)")

In [ ]:
# Comparacion visual de las tres estrategias sobre 'fare'.
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(fare, bins=40, color="#e45756"); ax[0].set_title("Original (con outliers)")
ax[1].hist(fare_wins, bins=40, color="#f0a500"); ax[1].set_title("Winsorizado (cap P1-P99)")
ax[2].hist(fare_log, bins=40, color="#4c78a8"); ax[2].set_title("Transformado (log1p)")
for a in ax: a.set_xlabel("valor"); a.set_ylabel("conteo")
plt.tight_layout(); plt.show()

## Resumen — tabla de referencia rapida

| Metodo | Tipo | Idea | Robusto a outliers | Usar cuando |
|---|---|---|---|---|
| **z-score** | estadistico | $|z|>3$ con media/desv. | no | datos ~normales, pocos extremos |
| **IQR / Tukey** | estadistico | fuera de $Q_1-1.5\,IQR$, $Q_3+1.5\,IQR$ | si | default univariado, base del boxplot |
| **z-score modif. (MAD)** | estadistico | mediana + MAD, $|M|>3.5$ | si | colas pesadas, criterio robusto |
| **Isolation Forest** | ML | aislamiento por cortes aleatorios | n/a | multivariado, datos grandes, score continuo |
| **DBSCAN** | ML | densidad: nucleos + ruido (`-1`) | n/a | clusters de forma arbitraria, anomalias globales |

**Que hacer despues:** *eliminar* si son errores y son pocos; *winsorizar* si son
reales pero distorsionan; *transformar* (log / Box-Cox) para comprimir colas sin
borrar informacion. La eleccion depende del origen del outlier, su cantidad y el
modelo.

> En el **Notebook 3** pasamos de la limpieza al **feature store** (Feast): como
> servir estas features de forma consistente entre entrenamiento y produccion.